#### 399. Evaluate Division
* https://leetcode.com/problems/evaluate-division/description/

#### BBG - IMP

In [ ]:
# Union Find - Preferred Solution
# TC - O(V), search operation - O(1) # almost constant
# SC - O(V)


class UnionFind:
    def __init__(self):
        self.parent = {}
        self.weight = {}
    
    def find(self, node):
        if node not in self.parent:
            self.parent[node] = node
            self.weight[node] = 1.0
            return node
        
        if node != self.parent[node]:
            orig_parent = self.parent[node]
            root = self.find(orig_parent)
            self.parent[node] = root
            # Path compression with weight update
            self.weight[node] *= self.weight[orig_parent]
        
        return self.parent[node]
    
    def union(self, nodex, nodey, value):
        rootx = self.find(nodex)
        rooty = self.find(nodey)

        if rootx != rooty:
            # Attach root_x under root_y
            self.parent[rootx] = rooty
             # Adjust weight
            # weight[root_x] * value_of_root_x = value * weight[root_y]
            self.weight[rootx] = value * self.weight[nodey] / self.weight[nodex]
    
    def connected(self, nodex, nodey):
        if nodex not in self.parent or nodey not in self.parent:
            return -1.0

        rootx = self.find(nodex)
        rooty = self.find(nodey)

        if rootx != rooty:
            return -1.0
        
        return self.weight[nodex] / self.weight[nodey]

class Solution:
    def calcEquation(self, equations: List[List[str]], values: List[float], queries: List[List[str]]) -> List[float]:

        uf = UnionFind()

        # Build graph
        for (x, y), val in zip(equations, values):
            uf.union(x, y, val)
        
        # Answer queries
        return [uf.connected(x, y) for x, y in queries]
        

In [ ]:
from typing import List, Dict, Deque, Set
import collections

class Solution:
    """
        Evaluates division queries using a weighted graph.

        Time Complexity:
            Graph construction: O(E)
            Each query (BFS): O(V + E) worst-case

        Space Complexity:
            O(V + E) for graph storage

        Graph - 
        consider a <-> b <-> c
        a -> b = 2, b -> a = 1/2
        b -> c = 3
        a -> c = 6

        Bidirectonal weighted graph

        Standard Graph + BFS (or DFS)
        create a graph - adjacency list
        mark the visited nodes  in a set
        for BFS use queue
    """
    def calcEquation(self, equations: List[List[str]], values: List[float], queries: List[List[str]]) -> List[float]:
        if not equations or not values or not queries:
            return []

        # create a graph
        graph: Dict[str, List] = collections.defaultdict(list)

        for (numerator, denominator), value in zip(equations, values):
            graph[numerator].append((denominator, value))
            graph[denominator].append((numerator, 1/value))

        def bfs(start, end):
            if start not in graph or end not in graph:
                return -1.0

            # neutral value 1 to start, any numerical multiplied with 1 remains 1
            queue: Deque[str, float] = collections.deque([(start, 1.0)])
            visited: Set[str] = set([start])

            while queue:
                current_node, current_product = queue.popleft()
                if current_node == end: # remember this
                    return current_product

                for nei, wt in graph[current_node]:
                    if nei not in visited:
                        visited.add(nei)
                        queue.append((nei, current_product*wt))

            return -1.0  

        return [bfs(start, end) for start, end in queries]

Solution().calcEquation(equations = [["a","b"],["b","c"]], values = [2.0,3.0], queries = [["a","c"],["b","a"],["a","e"],["a","a"],["x","x"]])
        

[6.0, 0.5, -1.0, 1.0, -1.0]